In [53]:
import requests
import os
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_core.messages import HumanMessage, SystemMessage
import json
from dotenv import load_dotenv
from tavily import TavilyClient

load_dotenv()

tavily_key = os.getenv("TAVILY_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

In [54]:
nfl_url = "https://www.nfl.com/news/all-news"
espn_url = "https://www.foxsports.com/nfl/news"

urls = {
    "nfl": nfl_url,
    "fox": espn_url
}

tavily_client = TavilyClient(api_key=tavily_key)

In [55]:
tavily_client = TavilyClient(api_key=tavily_key)

raw_content = {}
for source in urls:
    url = urls[source]
    response = tavily_client.extract(urls=[url])
    raw_content[source] = response['results'][0]['raw_content']  

In [56]:
import os
from typing import List, Optional
from datetime import datetime
from enum import Enum
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import PydanticOutputParser


class NFLTeam(str, Enum):
    ARIZONA_CARDINALS = "Arizona Cardinals"
    ATLANTA_FALCONS = "Atlanta Falcons"
    BALTIMORE_RAVENS = "Baltimore Ravens"
    BUFFALO_BILLS = "Buffalo Bills"
    CAROLINA_PANTHERS = "Carolina Panthers"
    CHICAGO_BEARS = "Chicago Bears"
    CINCINNATI_BENGALS = "Cincinnati Bengals"
    CLEVELAND_BROWNS = "Cleveland Browns"
    DALLAS_COWBOYS = "Dallas Cowboys"
    DENVER_BRONCOS = "Denver Broncos"
    DETROIT_LIONS = "Detroit Lions"
    GREEN_BAY_PACKERS = "Green Bay Packers"
    HOUSTON_TEXANS = "Houston Texans"
    INDIANAPOLIS_COLTS = "Indianapolis Colts"
    JACKSONVILLE_JAGUARS = "Jacksonville Jaguars"
    KANSAS_CITY_CHIEFS = "Kansas City Chiefs"
    LAS_VEGAS_RAIDERS = "Las Vegas Raiders"
    LOS_ANGELES_CHARGERS = "Los Angeles Chargers"
    LOS_ANGELES_RAMS = "Los Angeles Rams"
    MIAMI_DOLPHINS = "Miami Dolphins"
    MINNESOTA_VIKINGS = "Minnesota Vikings"
    NEW_ENGLAND_PATRIOTS = "New England Patriots"
    NEW_ORLEANS_SAINTS = "New Orleans Saints"
    NEW_YORK_GIANTS = "New York Giants"
    NEW_YORK_JETS = "New York Jets"
    PHILADELPHIA_EAGLES = "Philadelphia Eagles"
    PITTSBURGH_STEELERS = "Pittsburgh Steelers"
    SAN_FRANCISCO_49ERS = "San Francisco 49ers"
    SEATTLE_SEAHAWKS = "Seattle Seahawks"
    TAMPA_BAY_BUCCANEERS = "Tampa Bay Buccaneers"
    TENNESSEE_TITANS = "Tennessee Titans"
    WASHINGTON_COMMANDERS = "Washington Commanders"

# Define the news category enum
class NewsCategory(str, Enum):
    INJURY = "injury"
    ROSTER_MOVE = "roster_move"
    FANTASY_NEWS = "fantasy_news"
    TRADE = "trade"
    TEAM_NEWS = "team_news"
    PLAYER_UPDATE = "player_update"
    COACHING_CHANGE = "coaching_change"
    DRAFT_NEWS = "draft_news"
    CONTRACT_NEWS = "contract_news"
    SUSPENSION = "suspension"
    PERFORMANCE = "performance"

# Define the structure for individual news items
class NewsItem(BaseModel):
    title: str = Field(description="Brief headline of the news item")
    description: str = Field(description="Key details and context of the news")
    date: Optional[str] = Field(description="Date of the news if available (YYYY-MM-DD format)")
    impact_significance: str = Field(description="Analysis of the impact and significance, especially for fantasy football")
    label: NewsCategory = Field(description="Category/type of the news item")
    team: Optional[NFLTeam] = Field(description="Team if applicable")
    source: str = Field(description="Source of the news (e.g., nfl, espn, yahoo)")

# Define the main summary structure
class NFLNewsSummary(BaseModel):
    all_news: List[NewsItem] = Field(description="News from around the league")

def summarize_nfl_news(raw_content: str, source: str) -> NFLNewsSummary:
    # Initialize the LLM
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.1,
        api_key=os.getenv("OPENAI_API_KEY")
    )
    
    # Set up the Pydantic output parser
    parser = PydanticOutputParser(pydantic_object=NFLNewsSummary)
    
    # Create the system prompt with format instructions
    system_prompt = f"""You are an NFL news summarizer. Your job is to analyze raw NFL news content and create a clear, concise summary.

Please summarize the NFL news from the provided raw content according to the specified JSON structure.

IMPORTANT: Only include factual news items about actual NFL events, transactions, injuries, roster moves, or official announcements FROM THE LAST 24 HOURS. Do NOT include:
- Opinion pieces or editorials
- Speculation or rumors
- Fantasy football advice columns
- Power rankings or predictions
- Analysis pieces without concrete news
- Mock drafts or hypothetical scenarios
- News items that are older than 1 day

For each news item, you must include:
- title: Brief headline
- description: Key details and context
- date: Date if available (use YYYY-MM-DD format, or null if not available)
- impact_significance: Analysis of impact and significance, especially for fantasy football
- label: Choose from: injury, roster_move, trade, team_news, player_update, coaching_change, draft_news, contract_news, suspension, performance
- source: Set this to "{source}" for all news items (this indicates the news source)

Simply return all news items in a single list without categorizing them.

Focus on factual, concrete information from the last 24 hours only. Only include fantasy football impact analysis if the news item itself is factual. Keep summaries concise but informative.

{parser.get_format_instructions()}

Raw content to summarize: {raw_content}"""
    
    # Create messages
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content="Please analyze and summarize this NFL news content.")
    ]
    
    # Get response from LLM
    response = llm.invoke(messages)
    
    # Parse the response using Pydantic
    try:
        parsed_summary = parser.parse(response.content)
        return parsed_summary
    except Exception as e:
        print(f"Error parsing response: {e}")
        print(f"Raw response: {response.content}")
        raise

In [57]:
# Iterate through each source and its raw content
for source, content in raw_content.items():
    print(f"=== NFL NEWS SUMMARY - {source.upper()} ===\n")
    
    summary = summarize_nfl_news(content, source)
    
    for item in summary.all_news:
        print(f"• [{item.label.upper()}] [{item.team}] {item.title}")
        print(f"  Details: {item.description}")
        print(f"  Date: {item.date or 'Not specified'}")
        print(f"  Impact: {item.impact_significance}")
        print()
    
    print("\n" + "="*50 + "\n")

=== NFL NEWS SUMMARY - NFL ===

• [ROSTER_MOVE] [Cleveland Browns] Browns' Dillon Gabriel tabbed as backup QB with Shedeur Sanders at No. 3
  Details: Coach Kevin Stefanski selected rookie third-round pick Dillion Gabriel as Joe Flacco's backup on Tuesday after the initial 53-man roster was set.
  Date: 2025-08-26
  Impact: Gabriel's promotion to backup QB indicates the Browns' confidence in his abilities, which could impact the team's passing game if Flacco struggles or gets injured.

• [INJURY] [Arizona Cardinals] Cardinals first-round pick Walter Nolen (calf) to start season on PUP list
  Details: The Arizona Cardinals are placing rookie defensive lineman Walter Nolen (calf) on the PUP list to start the season.
  Date: 2025-08-26
  Impact: Nolen's absence will affect the Cardinals' defensive line depth and performance early in the season.

• [ROSTER_MOVE] [New England Patriots] Patriots cut 2022 first-round pick Cole Strange after three seasons
  Details: Once a much-scrutinized fir